Objectif:
01_inscriptions_preparees.csv
02_test_initial_prepare.csv
03_tests_intermediaires_prepares.csv
04_tests_finaux_prepares.csv
05_couverture_apres_preparation.csv
06_dataset_orientation_initiale.csv
07_dataset_prediction_finale_pre_scoring.csv
08_dataset_longitudinal_pre_scoring.csv

Cellule 1 — Importation des bibliothèques

In [1]:
# ============================================================
# ÉTAPE 1 — INTÉGRATION ET PRÉPARATION DES DONNÉES
# Projet : Prédiction et évaluation de la performance académique
# Objectif : harmoniser, nettoyer et préparer les sources
# ============================================================

import pandas as pd
import numpy as np
import re
import json
from pathlib import Path

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)

Cellule 2 — Définition des chemins

In [3]:
# ============================================================
# 1.1. Chemins des fichiers bruts
# ============================================================

DATA_DIR = Path("../data")

PATH_INSCRIPTIONS = DATA_DIR / "inscriptions.json"
PATH_TEST_INITIAL = DATA_DIR / "tests.json"
PATH_TESTS_INTERMEDIAIRES = DATA_DIR / "diego-mi-test.xlsx"
PATH_TESTS_FINAUX = DATA_DIR / "RESULTATS_FINAL_TESTS_PAR_PARCOURS.xlsx"

OUTPUT_DIR = Path("outputs/01_integration_preparation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

fichiers = {
    "inscriptions": PATH_INSCRIPTIONS,
    "test_initial": PATH_TEST_INITIAL,
    "tests_intermediaires": PATH_TESTS_INTERMEDIAIRES,
    "tests_finaux": PATH_TESTS_FINAUX
}

for nom, chemin in fichiers.items():
    print(f"{nom:25s} | existe = {chemin.exists()} | {chemin}")

inscriptions              | existe = True | ../data/inscriptions.json
test_initial              | existe = True | ../data/tests.json
tests_intermediaires      | existe = True | ../data/diego-mi-test.xlsx
tests_finaux              | existe = True | ../data/RESULTATS_FINAL_TESTS_PAR_PARCOURS.xlsx


Cellule 3 — Chargement des sources

In [4]:
# ============================================================
# 1.2. Chargement des sources
# ============================================================

df_inscriptions_raw = pd.read_json(PATH_INSCRIPTIONS)
df_test_initial_raw = pd.read_json(PATH_TEST_INITIAL)

dict_intermediaires_raw = pd.read_excel(PATH_TESTS_INTERMEDIAIRES, sheet_name=None)
dict_finaux_raw = pd.read_excel(PATH_TESTS_FINAUX, sheet_name=None)

print("Données chargées avec succès.")
print("Inscriptions :", df_inscriptions_raw.shape)
print("Test initial :", df_test_initial_raw.shape)
print("Feuilles tests intermédiaires :", list(dict_intermediaires_raw.keys()))
print("Feuilles tests finaux :", list(dict_finaux_raw.keys()))

Données chargées avec succès.
Inscriptions : (472, 19)
Test initial : (315, 25)
Feuilles tests intermédiaires : ['MI-TEST-DA', 'MI-TEST-BI', 'MI-TEST-DS', 'MI-TEST-IA']
Feuilles tests finaux : ['FINAL-TEST-DA', 'FINAL-TEST-BI', 'FINAL-TEST-DS', 'FINAL-TEST-IA']


Cellule 4 — Fonctions utilitaires

In [9]:
# ============================================================
# 1.3. Fonctions utilitaires 
# ============================================================

import re
import json
import numpy as np
import pandas as pd
from pandas.api.types import is_scalar


def nettoyer_nom_colonne(col):
    """
    Nettoie les noms de colonnes sans perdre leur signification.
    """
    col = str(col).strip()
    col = re.sub(r"\s+", " ", col)
    return col


def normaliser_identifiant(valeur):
    """
    Normalise l'identifiant apprenant.
    """
    # Cas valeur manquante simple
    if valeur is None:
        return np.nan
    
    # Cas valeur scalaire classique
    if is_scalar(valeur):
        try:
            if pd.isna(valeur):
                return np.nan
        except Exception:
            pass
        
        valeur = str(valeur).strip()
        
        if valeur.lower() in ["", "nan", "none", "nat", "null"]:
            return np.nan
        
        valeur = re.sub(r"\s+", "", valeur)
        valeur = valeur.upper()
        
        return valeur
    
    # Cas rare : liste, dict, array
    return np.nan


def nettoyer_texte(valeur):
    """
    Nettoie les chaînes de caractères.
    Fonction robuste contre les listes, dictionnaires et tableaux.
    """
    
    # Cas None
    if valeur is None:
        return np.nan
    
    # Cas scalaire : texte, nombre, date, booléen
    if is_scalar(valeur):
        try:
            if pd.isna(valeur):
                return np.nan
        except Exception:
            pass
        
        if isinstance(valeur, str):
            valeur = valeur.strip()
            valeur = re.sub(r"\s+", " ", valeur)
            
            if valeur.lower() in ["", "nan", "none", "nat", "null"]:
                return np.nan
            
            return valeur
        
        return valeur
    
    # Cas dictionnaire
    if isinstance(valeur, dict):
        try:
            return json.dumps(valeur, ensure_ascii=False)
        except Exception:
            return str(valeur)
    
    # Cas liste, tuple, set ou array
    if isinstance(valeur, (list, tuple, set, np.ndarray)):
        try:
            return json.dumps(list(valeur), ensure_ascii=False)
        except Exception:
            return str(valeur)
    
    # Autres cas
    return str(valeur)


def supprimer_colonnes_techniques(df):
    """
    Supprime les colonnes techniques inutiles pour l'analyse pédagogique.
    """
    colonnes_a_supprimer = [
        "UpdatedAt",
        "Commentaires",
        "nc_created_by",
        "nc_updated_by",
        "nc_order",
        "nc_row_meta",
        "Title"
    ]
    
    colonnes_existantes = [col for col in colonnes_a_supprimer if col in df.columns]
    
    return df.drop(columns=colonnes_existantes)


def extraire_parcours_depuis_nom_feuille(nom_feuille):
    """
    Extrait le parcours depuis le nom de feuille Excel.
    Exemples :
    - MI-TEST-DA -> DA
    - FINAL-TEST-BI -> BI
    """
    nom = str(nom_feuille).upper()
    
    for parcours in ["DA", "BI", "DS", "IA"]:
        if re.search(rf"(^|[-_ ]){parcours}($|[-_ ])", nom) or nom.endswith(parcours):
            return parcours
    
    return "INCONNU"


def afficher_shape(nom, df):
    print(f"{nom:40s} : {df.shape[0]} lignes | {df.shape[1]} colonnes")

Cellule 5 — Préparation générale d’un DataFrame

In [ ]:
# ============================================================
# 1.4. Fonction générale de préparation
# ============================================================

def preparer_dataframe_general(df, colonne_id_source=None):
    """
    Prépare une source :
    - nettoyage des noms de colonnes ;
    - harmonisation de la colonne IDENTIFICATION ;
    - nettoyage robuste des textes ;
    - suppression des colonnes techniques inutiles ;
    - séparation des lignes avec et sans identifiant.
    """
    
    df = df.copy()
    
    # Nettoyage des noms de colonnes
    df.columns = [nettoyer_nom_colonne(col) for col in df.columns]
    
    # Harmonisation de la colonne identifiant
    if colonne_id_source is not None and colonne_id_source in df.columns:
        df = df.rename(columns={colonne_id_source: "IDENTIFICATION"})
    elif "Identification" in df.columns:
        df = df.rename(columns={"Identification": "IDENTIFICATION"})
    elif "identification" in df.columns:
        df = df.rename(columns={"identification": "IDENTIFICATION"})
    
    if "IDENTIFICATION" not in df.columns:
        print("Colonnes disponibles :")
        print(list(df.columns))
        raise ValueError("Aucune colonne IDENTIFICATION trouvée dans cette source.")
    
    # Normalisation de l'identifiant
    df["IDENTIFICATION"] = df["IDENTIFICATION"].apply(normaliser_identifiant)
    
    # Nettoyage robuste des colonnes de type object
    colonnes_object = df.select_dtypes(include=["object"]).columns
    
    for col in colonnes_object:
        df[col] = df[col].map(nettoyer_texte)
    
    # Suppression des colonnes techniques
    df = supprimer_colonnes_techniques(df)
    
    # Séparation des lignes sans identifiant
    df_sans_id = df[df["IDENTIFICATION"].isna()].copy()
    df_avec_id = df[df["IDENTIFICATION"].notna()].copy()
    
    # Réinitialisation des index
    df_avec_id = df_avec_id.reset_index(drop=True)
    df_sans_id = df_sans_id.reset_index(drop=True)
    
    return df_avec_id, df_sans_id

Cellule 6 — Préparation des inscriptions

C’est cohérent avec l’audit, car les données d’inscription avaient 472 identifiants uniques, 0 identifiant manquant et 0 doublon d’identifiant.

In [11]:
# ============================================================
# 1.5. Préparation des données d'inscription
# ============================================================

df_inscriptions, df_inscriptions_sans_id = preparer_dataframe_general(
    df_inscriptions_raw,
    colonne_id_source="IDENTIFICATION"
)

afficher_shape("Inscriptions préparées", df_inscriptions)
afficher_shape("Inscriptions sans ID", df_inscriptions_sans_id)

display(df_inscriptions.head())

Inscriptions préparées                   : 472 lignes | 17 colonnes
Inscriptions sans ID                     : 0 lignes | 17 colonnes


,CreatedAt,IDENTIFICATION,Année de naissance,Genre,Région d'origine,Filière,Niveau d'étude,Statut actuel,Niveau informatique,Niveau Excel,Niveau Power BI,Niveau Python,Niveau IA,Motivation,Objectif pro,Disponibilité,Source info
0,2026-03-31 23:34:20+00:00,DIEG2026-001,2007,Masculin,Analamanga,Informatique / Genie logiciel,Licence 2 (L2),Etudiant(e) a plein temps,Intermediaire,Avance,Intermediaire,Intermediaire,Debutant,"[""Evoluer dans mon poste""]",Data Scientist,"[""En semaine - Matin""]",LinkedIn
1,2026-04-01 00:36:16+00:00,DIEG2026-002,20002,Masculin,Atsimo-Atsinanana,Polytechnique / Genie civil / Genie industriel,Master 1 (M1),Diplome(e) en recherche d emploi,Avance,Avance,Avance,Avance,Intermediaire,"[""Acquerir des competences techniques""]",Developpeur Python / IA,"[""En semaine - Apres-midi""]",Ami(e)
2,2026-04-01 03:01:14+00:00,DIEG2026-003,2001,Feminin,Diana,Droit / Sciences politiques,Master 2 (M2),Diplome(e) en recherche d emploi,Aucun,Aucun,Aucun,Aucun,Aucun,"[""Ameliorer mon employabilite"", "" Acquerir des...",Entrepreneur,"[""En semaine - Matin""]",Facebook
3,2026-04-01 03:32:24+00:00,DIEG2026-004,2003,Masculin,Diana,Autre,Master 2 (M2),Etudiant(e) en fin d etudes,Intermediaire,Intermediaire,Debutant,Intermediaire,Intermediaire,"[""Acquerir des competences techniques"", "" Amel...",Data Scientist,"[""Flexible""]",NaN
4,2026-04-01 06:14:56+00:00,DIEG2026-005,2003,Feminin,Analamanga,"EGS - Economie, Gestion & Sociologie",Licence 3 (L3),Diplome(e) en recherche d emploi,Intermediaire,Intermediaire,Aucun,Aucun,Debutant,"[""Acquerir des competences techniques"", "" Amel...",Business Analyst,"[""Flexible""]",Facebook


Cellule 7 — Correction des années de naissance

Correction des 9 années de naissance

In [ ]:
# ============================================================
# 1.6. Correction des années de naissance
# ============================================================

colonnes_annee = [
    col for col in df_inscriptions.columns
    if "naissance" in str(col).lower()
    or "annee" in str(col).lower()
    or "année" in str(col).lower()
]

print("Colonnes candidates année de naissance :", colonnes_annee)

if len(colonnes_annee) > 0:
    col_annee = colonnes_annee[0]
    
    df_inscriptions["annee_naissance_originale"] = df_inscriptions[col_annee]
    
    df_inscriptions["annee_naissance"] = pd.to_numeric(
        df_inscriptions[col_annee],
        errors="coerce"
    )
    
    # Valeurs aberrantes :
    # - non numériques
    # - inférieures à 1950
    # - supérieures à 2015
    masque_annee_aberrante = (
        df_inscriptions["annee_naissance"].isna()
        | (df_inscriptions["annee_naissance"] < 1950)
        | (df_inscriptions["annee_naissance"] > 2015)
    )
    
    df_annees_aberrantes = df_inscriptions.loc[
        masque_annee_aberrante,
        ["IDENTIFICATION", "annee_naissance_originale", "annee_naissance"]
    ].copy()
    
    # On neutralise les années aberrantes
    df_inscriptions.loc[masque_annee_aberrante, "annee_naissance"] = np.nan
    
    # Année de référence de l'étude
    ANNEE_REFERENCE = 2026
    
    df_inscriptions["age"] = ANNEE_REFERENCE - df_inscriptions["annee_naissance"]
    
    print("Colonne utilisée :", col_annee)
    print("Nombre d'années de naissance aberrantes neutralisées :", len(df_annees_aberrantes))
    
    display(df_annees_aberrantes.head(20))
    display(df_inscriptions[["IDENTIFICATION", "annee_naissance_originale", "annee_naissance", "age"]].head())

else:
    df_annees_aberrantes = pd.DataFrame()
    print("Aucune colonne année de naissance détectée.")

Colonnes candidates année de naissance : ['Année de naissance']
Colonne utilisée : Année de naissance
Nombre d'années de naissance aberrantes neutralisées : 9


,IDENTIFICATION,annee_naissance_originale,annee_naissance
1,DIEG2026-002,20002,20002
5,DIEG2026-006,20002000,20002000
18,DIEG2026-019,2020055555200,2020055555200
26,DIEG2026-027,200001022006,200001022006
31,DIEG2026-032,200000,200000
33,DIEG2026-034,8032002,8032002
102,DIEG2026-103,20002003,20002003
198,DIEG2026-199,21,21
395,DIEG2026-396,24,24


,IDENTIFICATION,annee_naissance_originale,annee_naissance,age
0,DIEG2026-001,2007,2007.0,19.0
1,DIEG2026-002,20002,NaN,NaN
2,DIEG2026-003,2001,2001.0,25.0
3,DIEG2026-004,2003,2003.0,23.0
4,DIEG2026-005,2003,2003.0,23.0


Cellule 8 — Préparation du test initial

In [13]:
# ============================================================
# 1.7. Préparation du test initial Q1-Q20
# ============================================================

df_test_initial, df_test_initial_sans_id = preparer_dataframe_general(
    df_test_initial_raw,
    colonne_id_source="Identification"
)

afficher_shape("Test initial préparé", df_test_initial)
afficher_shape("Test initial sans ID", df_test_initial_sans_id)

display(df_test_initial.head())

Test initial préparé                     : 305 lignes | 23 colonnes
Test initial sans ID                     : 10 lignes | 23 colonnes


,CreatedAt,Date du test,Q1 - Resume ventes Excel,Q2 - Import CSV Excel,Q3 - Modele donnees Excel,Q4 - Mauvaise pratique visu Excel,Q5 - 2e grande valeur Excel,Q6 - Mesure vs Colonne DAX,Q7 - CA annee precedente DAX,Q8 - Vue Modele Power BI,Q9 - Acces directeurs regionaux,Q10 - 12 commerciaux 3 indicateurs,Q11 - Bibliotheque CSV Python,Q12 - Overfitting Underfitting,Q13 - Segmentation 50000 clients,Q14 - Deployer modele Python API,Q15 - Valeurs manquantes 30pc,Q16 - Role system prompt LLM,Q17 - Assistant IA PDF financiers,Q18 - Role embedding dans RAG,Q19 - Agent IA selection outil,Q20 - Sortie fiable LLM tableau,IDENTIFICATION
0,2026-03-31 23:36:12+00:00,2026-03-31 23:36:11+00:00,A. Filtre automatique,C. Formules SI imbriquees,D. Formules RECHERCHEV,B. Limiter les couleurs,D. RANG(plage;2),C. Mesure=visualisation Colonne=stockee,B. CALCULATE(SUM SAMEPERIODLASTYEAR),D. Definir relations entre tables,C. Rapport separe par directeur,D. Radar ou tableau matriciel conditionnel,E. Je ne sais pas,C. Underfitting,C. Regression lineaire,C. FastAPI ou Flask + joblib ou pickle,D. Supprimer toutes les lignes,E. Je ne sais pas,B. RAG - indexer et recuperer dynamiquement,C. Compresser les fichiers PDF,B. Ordre dans lequel les outils sont listes,C. Augmenter la temperature,DIEG2026-001
1,2026-04-01 00:37:59+00:00,2026-04-01 00:38:00+00:00,C. Formule SOMME(),A. Macros VBA manuelles,C. Power Pivot avec relations,C. Etiquettes de donnees claires,C. MAX(SI(...)),C. Mesure=visualisation Colonne=stockee,E. Je ne sais pas,B. Publier sur Power BI Service,B. Row-Level Security (RLS),B. Graphique en secteurs,A. matplotlib,C. Underfitting,C. Regression lineaire,D. Envoyer par e-mail,D. Supprimer toutes les lignes,E. Je ne sais pas,B. RAG - indexer et recuperer dynamiquement,B. Authentifier utilisateur,E. Je ne sais pas,D. JSON structure avec schema defini,DIEG2026-002
2,2026-04-01 03:43:02+00:00,2026-04-01 03:42:58+00:00,B. Tableau croise dynamique (TCD),D. Power Query,C. Power Pivot avec relations,D. Graphique 3D explose,C. MAX(SI(...)),C. Mesure=visualisation Colonne=stockee,B. CALCULATE(SUM SAMEPERIODLASTYEAR),D. Definir relations entre tables,D. Parametres de rapport,B. Graphique en secteurs,D. numpy,D. Overfitting,A. Reseau de neurones recurrent,C. FastAPI ou Flask + joblib ou pickle,D. Supprimer toutes les lignes,D. Definir role et contraintes du modele,C. Copier-coller les PDFs dans le prompt,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,DIEG2026-004
3,2026-04-01 06:31:18+00:00,2026-04-01 06:31:17+00:00,D. Graphique en barres,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,A. NB.SI(plage;sup0),A. Aucune difference,A. DATEADD(-12 MONTH),A. Creer visuels interactifs,A. Filtrer manuellement les visuels,E. Je ne sais pas,A. matplotlib,A. Probleme de normalisation,A. Reseau de neurones recurrent,A. Reentrainer a chaque requete,B. Remplacer par 0,B. Entrainer sur nouvelles donnees,B. RAG - indexer et recuperer dynamiquement,C. Compresser les fichiers PDF,E. Je ne sais pas,A. Reduire longueur du prompt,DIEG2026-006
4,2026-04-01 06:34:56+00:00,2026-04-01 06:34:53+00:00,C. Formule SOMME(),D. Power Query,C. Power Pivot avec relations,D. Graphique 3D explose,E. Je ne sais pas,C. Mesure=visualisation Colonne=stockee,B. CALCULATE(SUM SAMEPERIODLASTYEAR),A. Creer visuels interactifs,B. Row-Level Security (RLS),B. Graphique en secteurs,E. Je ne sais pas,E. Je ne sais pas,B. K-Means Clustering,D. Envoyer par e-mail,C. Imputer avec mediane ou modele,D. Definir role et contraintes du modele,B. RAG - indexer et recuperer dynamiquement,C. Compresser les fichiers PDF,C. Raisonnement LLM (function calling),D. JSON structure avec schema defini,DIEG2026-005


Cellule 9 — Contrôle des questions Q1 à Q20

In [14]:
# ============================================================
# 1.8. Contrôle des colonnes Q1 à Q20 — VERSION CORRIGÉE
# ============================================================

def detecter_numero_question(colonne):
    """
    Détecte le numéro d'une question dans un nom de colonne.
    Exemples détectés :
    - Q1
    - Q1 - Excel
    - Q10 - Power BI
    - Q20 - Sortie fiable LLM tableau
    """
    col = str(colonne).strip().upper()
    
    match = re.match(r"^Q\s*([1-9]|1[0-9]|20)(\s|-|_|:|\.|$)", col)
    
    if match:
        return int(match.group(1))
    
    return None


def detecter_colonnes_questions_q1_q20(df):
    """
    Retourne un dictionnaire :
    {
        1: nom_colonne_Q1,
        2: nom_colonne_Q2,
        ...
        20: nom_colonne_Q20
    }
    """
    colonnes_questions = {}
    
    for col in df.columns:
        numero = detecter_numero_question(col)
        
        if numero is not None and 1 <= numero <= 20:
            colonnes_questions[numero] = col
    
    return colonnes_questions


# Détection des colonnes Q1 à Q20
colonnes_q_dict = detecter_colonnes_questions_q1_q20(df_test_initial)

# Liste ordonnée des colonnes détectées
colonnes_q_initial = [
    colonnes_q_dict[i]
    for i in range(1, 21)
    if i in colonnes_q_dict
]

questions_attendues = list(range(1, 21))
questions_presentes = sorted(list(colonnes_q_dict.keys()))
questions_absentes = [
    q for q in questions_attendues
    if q not in questions_presentes
]

print("Nombre de questions détectées :", len(colonnes_q_initial))
print("Questions présentes :", questions_presentes)
print("Questions absentes :", questions_absentes)

print("\nColonnes détectées :")
for numero, colonne in sorted(colonnes_q_dict.items()):
    print(f"Q{numero:02d} -> {colonne}")

Nombre de questions détectées : 20
Questions présentes : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
Questions absentes : []

Colonnes détectées :
Q01 -> Q1 - Resume ventes Excel
Q02 -> Q2 - Import CSV Excel
Q03 -> Q3 - Modele donnees Excel
Q04 -> Q4 - Mauvaise pratique visu Excel
Q05 -> Q5 - 2e grande valeur Excel
Q06 -> Q6 - Mesure vs Colonne DAX
Q07 -> Q7 - CA annee precedente DAX
Q08 -> Q8 - Vue Modele Power BI
Q09 -> Q9 - Acces directeurs regionaux
Q10 -> Q10 - 12 commerciaux 3 indicateurs
Q11 -> Q11 - Bibliotheque CSV Python
Q12 -> Q12 - Overfitting Underfitting
Q13 -> Q13 - Segmentation 50000 clients
Q14 -> Q14 - Deployer modele Python API
Q15 -> Q15 - Valeurs manquantes 30pc
Q16 -> Q16 - Role system prompt LLM
Q17 -> Q17 - Assistant IA PDF financiers
Q18 -> Q18 - Role embedding dans RAG
Q19 -> Q19 - Agent IA selection outil
Q20 -> Q20 - Sortie fiable LLM tableau


Cellule 10 corrigée — Préparation des tests intermédiaires

In [15]:
# ============================================================
# 1.9. Préparation et concaténation des tests intermédiaires
# VERSION CORRIGÉE ET STANDARDISÉE
# ============================================================

def standardiser_questions_intermediaires(df, prefixe="mi"):
    """
    Standardise les colonnes de questions des tests intermédiaires.
    
    Les questions propres à chaque parcours ont des intitulés différents.
    Pour faciliter la concaténation, on les renomme :
    mi_Q01, mi_Q02, ..., mi_Q15.
    """
    
    df = df.copy()
    
    colonnes_meta = {
        "IDENTIFICATION",
        "Id",
        "CreatedAt",
        "UpdatedAt",
        "InscriptionId",
        "RowId",
        "Filiere",
        "Filière",
        "Niveau d'Etude",
        "Niveau d'étude",
        "parcours_test_intermediaire",
        "feuille_source_intermediaire"
    }
    
    colonnes_questions = [
        col for col in df.columns
        if col not in colonnes_meta
    ]
    
    mapping_questions = {}
    
    for i, col in enumerate(colonnes_questions, start=1):
        mapping_questions[col] = f"{prefixe}_Q{i:02d}"
    
    df = df.rename(columns=mapping_questions)
    
    return df, mapping_questions


liste_intermediaires = []
liste_intermediaires_sans_id = []
liste_mapping_questions_intermediaires = []

for nom_feuille, df_raw in dict_intermediaires_raw.items():
    
    print("\nTraitement de la feuille :", nom_feuille)
    
    parcours = extraire_parcours_depuis_nom_feuille(nom_feuille)
    
    # Préparation générale : nettoyage, ID, colonnes techniques, lignes sans ID
    df_temp, df_temp_sans_id = preparer_dataframe_general(
        df_raw,
        colonne_id_source="IDENTIFICATION"
    )
    
    # Standardisation des questions
    df_temp, mapping_questions = standardiser_questions_intermediaires(
        df_temp,
        prefixe="mi"
    )
    
    # Même renommage pour les lignes sans identifiant
    df_temp_sans_id = df_temp_sans_id.rename(columns=mapping_questions)
    
    # Ajout des informations de parcours et de feuille source
    df_temp["parcours_test_intermediaire"] = parcours
    df_temp["feuille_source_intermediaire"] = nom_feuille
    
    df_temp_sans_id["parcours_test_intermediaire"] = parcours
    df_temp_sans_id["feuille_source_intermediaire"] = nom_feuille
    
    # Sauvegarde du mapping des questions
    for ancienne_colonne, nouvelle_colonne in mapping_questions.items():
        liste_mapping_questions_intermediaires.append({
            "feuille": nom_feuille,
            "parcours": parcours,
            "ancienne_colonne": ancienne_colonne,
            "nouvelle_colonne": nouvelle_colonne
        })
    
    liste_intermediaires.append(df_temp)
    liste_intermediaires_sans_id.append(df_temp_sans_id)
    
    print("Parcours détecté :", parcours)
    print("Lignes avec ID :", df_temp.shape[0])
    print("Lignes sans ID :", df_temp_sans_id.shape[0])
    print("Nombre de questions standardisées :", len(mapping_questions))


# Concaténation finale
df_tests_intermediaires = pd.concat(
    liste_intermediaires,
    ignore_index=True
)

df_tests_intermediaires_sans_id = pd.concat(
    liste_intermediaires_sans_id,
    ignore_index=True
)

df_mapping_questions_intermediaires = pd.DataFrame(
    liste_mapping_questions_intermediaires
)

afficher_shape("Tests intermédiaires préparés", df_tests_intermediaires)
afficher_shape("Tests intermédiaires sans ID", df_tests_intermediaires_sans_id)
afficher_shape("Mapping questions intermédiaires", df_mapping_questions_intermediaires)

print("\nRépartition par parcours :")
display(df_tests_intermediaires["parcours_test_intermediaire"].value_counts())

print("\nAperçu des tests intermédiaires préparés :")
display(df_tests_intermediaires.head())

print("\nMapping des questions intermédiaires :")
display(df_mapping_questions_intermediaires.head(20))


Traitement de la feuille : MI-TEST-DA
Parcours détecté : DA
Lignes avec ID : 32
Lignes sans ID : 5
Nombre de questions standardisées : 15

Traitement de la feuille : MI-TEST-BI
Parcours détecté : BI
Lignes avec ID : 60
Lignes sans ID : 5
Nombre de questions standardisées : 15

Traitement de la feuille : MI-TEST-DS
Parcours détecté : DS
Lignes avec ID : 44
Lignes sans ID : 2
Nombre de questions standardisées : 15

Traitement de la feuille : MI-TEST-IA
Parcours détecté : IA
Lignes avec ID : 60
Lignes sans ID : 2
Nombre de questions standardisées : 15
Tests intermédiaires préparés            : 196 lignes | 20 colonnes
Tests intermédiaires sans ID             : 14 lignes | 20 colonnes
Mapping questions intermédiaires         : 60 lignes | 4 colonnes

Répartition par parcours :


parcours_test_intermediaire
BI    60
IA    60
DS    44
DA    32
Name: count, dtype: int64


Aperçu des tests intermédiaires préparés :


,IDENTIFICATION,Id,CreatedAt,mi_Q01,mi_Q02,mi_Q03,mi_Q04,mi_Q05,mi_Q06,mi_Q07,mi_Q08,mi_Q09,mi_Q10,mi_Q11,mi_Q12,mi_Q13,mi_Q14,mi_Q15,parcours_test_intermediaire,feuille_source_intermediaire
0,DIEG2026-463,17,2026-05-28 11:55:09+00:00,"C) Garbage In, Garbage Out — des donnees sourc...",A) Completude — il manque des valeurs,"B) Nettoyer, filtrer, enrichir et formater les...",B) Cliquer sur Transformer les donnees pour ou...,B) C'est une limitation de la licence Excel St...,C) Les deux produisent exactement le meme resu...,B) Table.SelectColumns liste uniquement les co...,C) Le zero initial est perdu — la valeur devie...,C) Connecter > Nommer > Filtrer > Supprimer co...,B) Avant chaque operation de nettoyage pour ev...,"C) Au centre, reliee a toutes les tables de di...",D) Elle doit s'appeler ID ou Code par convention,C) La ligne DIR n'a aucune correspondance — se...,A) Inner Join — uniquement les lignes correspo...,A) Parce que Power Pivot ne peut pas charger u...,DA,MI-TEST-DA
1,DIEG2026-636,18,2026-05-28 11:56:26+00:00,"C) Garbage In, Garbage Out — des donnees sourc...",D) Exactitude — la valeur ne represente pas co...,A) Recuperer les donnees depuis les sources (C...,B) Cliquer sur Transformer les donnees pour ou...,C) Il utilise l'evaluation paresseuse (Lazy Ev...,B) Dupliquer cree une copie independante (sour...,B) Table.SelectColumns liste uniquement les co...,C) Le zero initial est perdu — la valeur devie...,C) Connecter > Nommer > Filtrer > Supprimer co...,"C) En toute derniere etape de la requete, apre...","C) Au centre, reliee a toutes les tables de di...",B) Elle doit etre strictement unique — aucun d...,C) La ligne DIR n'a aucune correspondance — se...,D) Full Outer Join — toutes les lignes des deu...,B) Parce que la mega-table duplique les donnee...,DA,MI-TEST-DA
2,DIEG2026-633,20,2026-05-28 12:01:43+00:00,"C) Garbage In, Garbage Out — des donnees sourc...",D) Exactitude — la valeur ne represente pas co...,"B) Nettoyer, filtrer, enrichir et formater les...",B) Cliquer sur Transformer les donnees pour ou...,D) Les lignes supplementaires sont automatique...,B) Dupliquer cree une copie independante (sour...,D) Il n'y a aucune difference pratique entre l...,C) Le zero initial est perdu — la valeur devie...,D) Connecter > Charger > Transformer les donne...,"C) En toute derniere etape de la requete, apre...","C) Au centre, reliee a toutes les tables de di...",B) Elle doit etre strictement unique — aucun d...,B) La relation fonctionne mais avec une legere...,A) Inner Join — uniquement les lignes correspo...,B) Parce que la mega-table duplique les donnee...,DA,MI-TEST-DA
3,DIEG2026-442,21,2026-05-28 12:04:58+00:00,"C) Garbage In, Garbage Out — des donnees sourc...",A) Completude — il manque des valeurs,"B) Nettoyer, filtrer, enrichir et formater les...",B) Cliquer sur Transformer les donnees pour ou...,B) C'est une limitation de la licence Excel St...,B) Dupliquer cree une copie independante (sour...,B) Table.SelectColumns liste uniquement les co...,C) Le zero initial est perdu — la valeur devie...,D) Connecter > Charger > Transformer les donne...,B) Avant chaque operation de nettoyage pour ev...,"C) Au centre, reliee a toutes les tables de di...",B) Elle doit etre strictement unique — aucun d...,B) La relation fonctionne mais avec une legere...,A) Inner Join — uniquement les lignes correspo...,D) Parce que le schema en etoile est une exige...,DA,MI-TEST-DA
4,DIEG2026-055,23,2026-05-28 12:14:16+00:00,"C) Garbage In, Garbage Out — des donnees sourc...",B) Coherence — les donnees se contredisent,A) Recuperer les donnees depuis les sources (C...,B) Cliquer sur Transformer les donnees pour ou...,A) Power Query ne peut pas traiter plus de 1 0...,B) Dupliquer cree une copie independante (sour...,B) Table.SelectColumns liste uniquement les co...,C) Le zero initial est perdu — la valeur devie...,D) Connecter > Charger > Transformer les donne...,B) Avant chaque operation de nettoyage pour ev...,"C) Au centre, reliee a toutes les ta


Mapping des questions intermédiaires :


,feuille,parcours,ancienne_colonne,nouvelle_colonne
0,MI-TEST-DA,DA,Qu'exprime le principe GIGO en Data Analysis ?,mi_Q01
1,MI-TEST-DA,DA,"Parmi les 4 piliers de la qualite des donnees,...",mi_Q02
2,MI-TEST-DA,DA,"Dans le processus ETL, l'etape Transform corre...",mi_Q03
3,MI-TEST-DA,DA,"Dans Power Query, quelle action doit-on toujou...",mi_Q04
4,MI-TEST-DA,DA,Pourquoi Power Query affiche-t-il uniquement ~...,mi_Q05
5,MI-TEST-DA,DA,Quelle difference fondamentale existe entre Du...,mi_Q06
6,MI-TEST-DA,DA,Pourquoi preferer Supprimer les autres colonne...,mi_Q07
7,MI-TEST-DA,DA,Une colonne Code Postal contient la valeur 060...,mi_Q08
8,MI-TEST-DA,DA,Quel est l'ordre correct du workflow dans Powe...,mi_Q09
9,MI-TEST-DA,DA,"Dans Power Query, a quel moment doit-on typer ...",mi_Q10


Cellule 11 corrigée — Préparation des tests finaux

In [16]:
# ============================================================
# 1.10. Préparation et concaténation des tests finaux
# VERSION CORRIGÉE ET STANDARDISÉE
# ============================================================

def standardiser_questions_finales(df, prefixe="final"):
    """
    Standardise les colonnes de questions des tests finaux.
    
    Les questions propres à chaque parcours ont des intitulés différents.
    Pour faciliter la concaténation, on les renomme :
    final_Q01, final_Q02, ..., final_Q15.
    """
    
    df = df.copy()
    
    colonnes_meta = {
        "IDENTIFICATION",
        "Id",
        "RowId",
        "CreatedAt",
        "UpdatedAt",
        "InscriptionId",
        "Filiere",
        "Filière",
        "Niveau d'Etude",
        "Niveau d'étude",
        "parcours_test_final",
        "feuille_source_finale"
    }
    
    colonnes_questions = [
        col for col in df.columns
        if col not in colonnes_meta
    ]
    
    mapping_questions = {}
    
    for i, col in enumerate(colonnes_questions, start=1):
        mapping_questions[col] = f"{prefixe}_Q{i:02d}"
    
    df = df.rename(columns=mapping_questions)
    
    return df, mapping_questions


liste_finaux = []
liste_finaux_sans_id = []
liste_mapping_questions_finales = []

for nom_feuille, df_raw in dict_finaux_raw.items():
    
    print("\nTraitement de la feuille :", nom_feuille)
    
    parcours = extraire_parcours_depuis_nom_feuille(nom_feuille)
    
    # Préparation générale : nettoyage, ID, colonnes techniques, lignes sans ID
    df_temp, df_temp_sans_id = preparer_dataframe_general(
        df_raw,
        colonne_id_source="IDENTIFICATION"
    )
    
    # Standardisation des questions finales
    df_temp, mapping_questions = standardiser_questions_finales(
        df_temp,
        prefixe="final"
    )
    
    # Même renommage pour les éventuelles lignes sans identifiant
    df_temp_sans_id = df_temp_sans_id.rename(columns=mapping_questions)
    
    # Ajout des informations de parcours et de feuille source
    df_temp["parcours_test_final"] = parcours
    df_temp["feuille_source_finale"] = nom_feuille
    
    df_temp_sans_id["parcours_test_final"] = parcours
    df_temp_sans_id["feuille_source_finale"] = nom_feuille
    
    # Sauvegarde du mapping des questions
    for ancienne_colonne, nouvelle_colonne in mapping_questions.items():
        liste_mapping_questions_finales.append({
            "feuille": nom_feuille,
            "parcours": parcours,
            "ancienne_colonne": ancienne_colonne,
            "nouvelle_colonne": nouvelle_colonne
        })
    
    liste_finaux.append(df_temp)
    liste_finaux_sans_id.append(df_temp_sans_id)
    
    print("Parcours détecté :", parcours)
    print("Lignes avec ID :", df_temp.shape[0])
    print("Lignes sans ID :", df_temp_sans_id.shape[0])
    print("Nombre de questions standardisées :", len(mapping_questions))


# Concaténation finale
df_tests_finaux = pd.concat(
    liste_finaux,
    ignore_index=True
)

df_tests_finaux_sans_id = pd.concat(
    liste_finaux_sans_id,
    ignore_index=True
)

df_mapping_questions_finales = pd.DataFrame(
    liste_mapping_questions_finales
)

afficher_shape("Tests finaux préparés", df_tests_finaux)
afficher_shape("Tests finaux sans ID", df_tests_finaux_sans_id)
afficher_shape("Mapping questions finales", df_mapping_questions_finales)

print("\nRépartition par parcours :")
display(df_tests_finaux["parcours_test_final"].value_counts())

print("\nAperçu des tests finaux préparés :")
display(df_tests_finaux.head())

print("\nMapping des questions finales :")
display(df_mapping_questions_finales.head(20))


Traitement de la feuille : FINAL-TEST-DA
Parcours détecté : DA
Lignes avec ID : 34
Lignes sans ID : 0
Nombre de questions standardisées : 15

Traitement de la feuille : FINAL-TEST-BI
Parcours détecté : BI
Lignes avec ID : 60
Lignes sans ID : 0
Nombre de questions standardisées : 15

Traitement de la feuille : FINAL-TEST-DS
Parcours détecté : DS
Lignes avec ID : 48
Lignes sans ID : 0
Nombre de questions standardisées : 15

Traitement de la feuille : FINAL-TEST-IA
Parcours détecté : IA
Lignes avec ID : 64
Lignes sans ID : 0
Nombre de questions standardisées : 15
Tests finaux préparés                    : 206 lignes | 23 colonnes
Tests finaux sans ID                     : 0 lignes | 23 colonnes
Mapping questions finales                : 60 lignes | 4 colonnes

Répartition par parcours :


parcours_test_final
IA    64
BI    60
DS    48
DA    34
Name: count, dtype: int64


Aperçu des tests finaux préparés :


,RowId,CreatedAt,InscriptionId,IDENTIFICATION,Filiere,Niveau d'Etude,final_Q01,final_Q02,final_Q03,final_Q04,final_Q05,final_Q06,final_Q07,final_Q08,final_Q09,final_Q10,final_Q11,final_Q12,final_Q13,final_Q14,final_Q15,parcours_test_final,feuille_source_finale
0,56,2026-06-06 17:08:28+00:00,55,DIEG2026-055,IST - Informatique & Sciences des Technologies,Licence 2 (L2),D) Changer le type de la colonne en Date avec ...,B) Remplacer les valeurs null par A traiter da...,D) Fusionner sur CodeClient puis developper Re...,B) Colonne conditionnelle,B) Fact_Factures,D) Agences 1 vers Transactions plusieurs,A) Une colonne calculee,A) Nb = COUNTROWS(Reclamations),"A) IF([Inscrits] = 0, 0, [Admis] / [Inscrits])...","A) CALCULATE(Filiere[Nom] = ""Informatique"", [T...",A) Barres groupees par filiere et par semaine,B) Filtre automatique applique a la source ava...,B) Barres empilees a 100 % pour voir les propo...,A) Lister les visuels possibles,B) Exclure les absences non justifiees pour re...,DA,FINAL-TEST-DA
1,28,2026-06-04 21:08:44+00:00,117,DIEG2026-117,"EGS - Economie, Gestion & Sociologie",Licence 1 (L1),A) Trier la colonne manuellement avant chaque ...,B) Remplacer les valeurs null par A traiter da...,C) Creer une relation sans importer Region dan...,C) Colonne a partir d'exemples avec quelques c...,D) Une table produit avec une ligne par facture,D) Agences 1 vers Transactions plusieurs,D) Un segment cree avant la colonne,B) Nb = COUNT(Reclamations[Id]) si Id est touj...,C) [Admis] / DISTINCTCOUNT(Etudiants[Id]),"C) CALCULATE([Total Montant], Filiere[Nom] = ""...",C) Courbe,D) Filtre de rapport regle separement dans cha...,C) Deux graphiques separes avec la meme echelle,A) Lister les visuels possibles,B) Exclure les absences non justifiees pour re...,DA,FINAL-TEST-DA
2,43,2026-06-05 07:52:11+00:00,195,DIEG2026-195,"EGS - Economie, Gestion & Sociologie",Licence 2 (L2),A) Trier la colonne manuellement avant chaque ...,A) Filtrer les lignes null puis les saisir dan...,C) Creer une relation sans importer Region dan...,B) Colonne conditionnelle,C) Une table calendrier contenant aussi les mo...,A) Transactions 1 vers Agences 1 si les noms d...,C) Une mesure car elle change selon le visuel,B) Nb = COUNT(Reclamations[Id]) si Id est touj...,B) CALCULATE([Admis] / [Inscrits]) sans autre ...,"C) CALCULATE([Total Montant], Filiere[Nom] = ""...",C) Courbe,A) Segment connecte aux TCD,A) Barres groupees,A) Lister les visuels possibles,"A) Segmenter par groupe, periode et module ava...",DA,FINAL-TEST-DA
3,35,2026-06-05 05:36:07+00:00,278,DIEG2026-278,IST - Informatique & Sciences des Technologies,Licence 1 (L1),C) Creer une colonne texte AAAA-MM-JJ sans con...,B) Remplacer les valeurs null par A traiter da...,D) Fusionner sur CodeClient puis developper Re...,B) Colonne conditionnelle,B) Fact_Factures,D) Agences 1 vers Transactions plusieurs,A) Une colonne calculee,A) Nb = COUNTROWS(Reclamations),"D) DIVIDE([Admis], [Inscrits])","C) CALCULATE([Total Montant], Filiere[Nom] = ""...",C) Courbe,A) Segment connecte aux TCD,A) Barres groupees,D) Definir les questions metier et les indicat...,"A) Segmenter par groupe, periode et module ava...",DA,FINAL-TEST-DA
4,31,2026-06-05 01:40:26+00:00,288,DIEG2026-288,Polytechnique / Genie civil / Genie industriel,Licence 1 (L1),D) Changer le type de la colonne en Date avec ...,B) Remplacer les valeurs null par A traiter da...,D) Fusionner sur CodeClient puis developper Re...,B) Colonne conditionnelle,B) Fact_Factures,D) Agences 1 vers Transactions plusieurs,A) Une colonne calculee,A) Nb = COUNTROWS(Reclamations),C) [Admis] / DISTINCTCOUNT(Etudiants[Id]),"C) CALCULATE([Total Montant], Filiere[Nom] = ""...",C) Courbe,A) Segment connecte aux TCD,A) Barres groupees,D) Definir les questions metier et les indicat...,"A) Segmenter par groupe, periode et module ava...",DA,FINAL-TEST-DA



Mapping des questions finales :


,feuille,parcours,ancienne_colonne,nouvelle_colonne
0,FINAL-TEST-DA,DA,"Dans Power Query, une colonne Date_Commande co...",final_Q01
1,FINAL-TEST-DA,DA,Une table de suivi contient des cellules null ...,final_Q02
2,FINAL-TEST-DA,DA,"Vous avez une table Ventes(CodeClient, Montant...",final_Q03
3,FINAL-TEST-DA,DA,"Dans Power Query, vous devez creer une colonne...",final_Q04
4,FINAL-TEST-DA,DA,"Dans un modele Power Pivot de facturation, que...",final_Q05
5,FINAL-TEST-DA,DA,Une table Agences contient une ligne par agenc...,final_Q06
6,FINAL-TEST-DA,DA,Vous devez stocker une categorie Age_Classe ca...,final_Q07
7,FINAL-TEST-DA,DA,Quelle mesure DAX compte le nombre de lignes d...,final_Q08
8,FINAL-TEST-DA,DA,Vous calculez Taux_Reussite = [Admis] / [Inscr...,final_Q09
9,FINAL-TEST-DA,DA,Vous avez [Total Montant] et une dimension Fil...,final_Q10


Cellule 12 — Vérification des doublons après préparation

In [17]:
# ============================================================
# 1.11. Vérification des doublons après préparation
# VERSION CORRIGÉE
# ============================================================

def resume_identifiants(df, nom_source):
    """
    Résume la qualité des identifiants après préparation.
    """
    if "IDENTIFICATION" not in df.columns:
        return {
            "source": nom_source,
            "nb_lignes": len(df),
            "nb_identifiants_uniques": np.nan,
            "nb_identifiants_manquants": np.nan,
            "nb_doublons_identifiants": np.nan,
            "statut": "Colonne IDENTIFICATION absente"
        }
    
    ids = df["IDENTIFICATION"]
    
    nb_lignes = len(df)
    nb_ids_uniques = ids.dropna().nunique()
    nb_ids_manquants = ids.isna().sum()
    nb_doublons = ids.dropna().duplicated().sum()
    
    statut = "OK" if nb_ids_manquants == 0 and nb_doublons == 0 else "À vérifier"
    
    return {
        "source": nom_source,
        "nb_lignes": nb_lignes,
        "nb_identifiants_uniques": nb_ids_uniques,
        "nb_identifiants_manquants": nb_ids_manquants,
        "nb_doublons_identifiants": nb_doublons,
        "statut": statut
    }


df_resume_ids_step1 = pd.DataFrame([
    resume_identifiants(df_inscriptions, "Inscriptions préparées"),
    resume_identifiants(df_test_initial, "Test initial préparé"),
    resume_identifiants(df_tests_intermediaires, "Tests intermédiaires préparés"),
    resume_identifiants(df_tests_finaux, "Tests finaux préparés")
])

display(df_resume_ids_step1)

,source,nb_lignes,nb_identifiants_uniques,nb_identifiants_manquants,nb_doublons_identifiants,statut
0,Inscriptions préparées,472,472,0,0,OK
1,Test initial préparé,305,305,0,0,OK
2,Tests intermédiaires préparés,196,196,0,0,OK
3,Tests finaux préparés,206,206,0,0,OK


Cellule 12 bis — Détail des doublons éventuels

In [18]:
# ============================================================
# 1.11 bis. Détail des identifiants dupliqués éventuels
# ============================================================

def extraire_doublons_apres_preparation(df, nom_source):
    """
    Extrait les lignes ayant des identifiants dupliqués.
    """
    if "IDENTIFICATION" not in df.columns:
        return pd.DataFrame()
    
    masque_doublons = (
        df["IDENTIFICATION"].notna()
        & df["IDENTIFICATION"].duplicated(keep=False)
    )
    
    df_doublons = df.loc[masque_doublons].copy()
    df_doublons["source"] = nom_source
    
    return df_doublons


liste_doublons_step1 = [
    extraire_doublons_apres_preparation(df_inscriptions, "Inscriptions préparées"),
    extraire_doublons_apres_preparation(df_test_initial, "Test initial préparé"),
    extraire_doublons_apres_preparation(df_tests_intermediaires, "Tests intermédiaires préparés"),
    extraire_doublons_apres_preparation(df_tests_finaux, "Tests finaux préparés")
]

df_doublons_step1 = pd.concat(
    [df for df in liste_doublons_step1 if not df.empty],
    ignore_index=True
) if any(not df.empty for df in liste_doublons_step1) else pd.DataFrame()

print("Nombre total de lignes avec identifiants dupliqués :", len(df_doublons_step1))

if not df_doublons_step1.empty:
    display(df_doublons_step1[["source", "IDENTIFICATION"]].sort_values(["source", "IDENTIFICATION"]))
else:
    print("Aucun doublon d'identifiant détecté après préparation.")

Nombre total de lignes avec identifiants dupliqués : 0
Aucun doublon d'identifiant détecté après préparation.


Cellule 13 corrigée — Préfixage des colonnes avant fusion

In [19]:
# ============================================================
# 1.12. Préfixage des colonnes avant fusion
# VERSION CORRIGÉE
# ============================================================

def prefixer_colonnes(df, prefixe, colonnes_sans_prefixe=("IDENTIFICATION",)):
    """
    Ajoute un préfixe aux colonnes pour éviter les collisions lors des fusions.
    
    Règles :
    - IDENTIFICATION n'est jamais préfixée ;
    - les colonnes qui commencent déjà par le préfixe ne sont pas repréfixées ;
    - les autres colonnes reçoivent le préfixe indiqué.
    """
    
    df = df.copy()
    mapping = {}
    
    for col in df.columns:
        col_str = str(col)
        
        if col_str in colonnes_sans_prefixe:
            continue
        
        if col_str.startswith(prefixe):
            continue
        
        mapping[col] = f"{prefixe}{col_str}"
    
    df = df.rename(columns=mapping)
    
    return df


# Préfixage des quatre sources préparées
df_inscriptions_pref = prefixer_colonnes(df_inscriptions, "ins_")
df_test_initial_pref = prefixer_colonnes(df_test_initial, "init_")
df_intermediaires_pref = prefixer_colonnes(df_tests_intermediaires, "mi_")
df_finaux_pref = prefixer_colonnes(df_tests_finaux, "final_")


print("Préfixage terminé.")

print("\nColonnes inscriptions préfixées :")
print(list(df_inscriptions_pref.columns[:15]))

print("\nColonnes test initial préfixées :")
print(list(df_test_initial_pref.columns[:15]))

print("\nColonnes tests intermédiaires préfixées :")
print(list(df_intermediaires_pref.columns[:15]))

print("\nColonnes tests finaux préfixées :")
print(list(df_finaux_pref.columns[:15]))


# Vérification rapide
print("\nVérification présence IDENTIFICATION :")
print("Inscriptions :", "IDENTIFICATION" in df_inscriptions_pref.columns)
print("Test initial :", "IDENTIFICATION" in df_test_initial_pref.columns)
print("Tests intermédiaires :", "IDENTIFICATION" in df_intermediaires_pref.columns)
print("Tests finaux :", "IDENTIFICATION" in df_finaux_pref.columns)

Préfixage terminé.

Colonnes inscriptions préfixées :
['ins_CreatedAt', 'IDENTIFICATION', 'ins_Année de naissance', 'ins_Genre', "ins_Région d'origine", 'ins_Filière', "ins_Niveau d'étude", 'ins_Statut actuel', 'ins_Niveau informatique', 'ins_Niveau Excel', 'ins_Niveau Power BI', 'ins_Niveau Python', 'ins_Niveau IA', 'ins_Motivation', 'ins_Objectif pro']

Colonnes test initial préfixées :
['init_CreatedAt', 'init_Date du test', 'init_Q1 - Resume ventes Excel', 'init_Q2 - Import CSV Excel', 'init_Q3 - Modele donnees Excel', 'init_Q4 - Mauvaise pratique visu Excel', 'init_Q5 - 2e grande valeur Excel', 'init_Q6 - Mesure vs Colonne DAX', 'init_Q7 - CA annee precedente DAX', 'init_Q8 - Vue Modele Power BI', 'init_Q9 - Acces directeurs regionaux', 'init_Q10 - 12 commerciaux 3 indicateurs', 'init_Q11 - Bibliotheque CSV Python', 'init_Q12 - Overfitting Underfitting', 'init_Q13 - Segmentation 50000 clients']

Colonnes tests intermédiaires préfixées :
['IDENTIFICATION', 'mi_Id', 'mi_CreatedAt', 

Cellule 14 corrigée — Dataset d’orientation initiale
Ce dataset fusionne:Inscriptions préparées + Test initial préparé

In [20]:
# ============================================================
# 1.13. Construction du dataset d'orientation initiale
# Inscriptions + test initial
# VERSION CORRIGÉE ET CONTRÔLÉE
# ============================================================

# Vérification avant fusion
print("Nombre d'ID inscriptions :", df_inscriptions_pref["IDENTIFICATION"].nunique())
print("Nombre d'ID test initial :", df_test_initial_pref["IDENTIFICATION"].nunique())

ids_communs_orientation = set(df_inscriptions_pref["IDENTIFICATION"]) & set(df_test_initial_pref["IDENTIFICATION"])

print("Nombre d'ID communs inscriptions + test initial :", len(ids_communs_orientation))


# Fusion contrôlée
df_dataset_orientation_initiale = df_inscriptions_pref.merge(
    df_test_initial_pref,
    on="IDENTIFICATION",
    how="inner",
    validate="one_to_one"
)

afficher_shape("Dataset orientation initiale", df_dataset_orientation_initiale)

print("\nVérification après fusion :")
print("Nombre de lignes :", len(df_dataset_orientation_initiale))
print("Nombre d'ID uniques :", df_dataset_orientation_initiale["IDENTIFICATION"].nunique())
print("Doublons ID :", df_dataset_orientation_initiale["IDENTIFICATION"].duplicated().sum())
print("Valeurs manquantes ID :", df_dataset_orientation_initiale["IDENTIFICATION"].isna().sum())

print("\nAperçu du dataset d'orientation initiale :")
display(df_dataset_orientation_initiale.head())

Nombre d'ID inscriptions : 472
Nombre d'ID test initial : 305
Nombre d'ID communs inscriptions + test initial : 305
Dataset orientation initiale             : 305 lignes | 42 colonnes

Vérification après fusion :
Nombre de lignes : 305
Nombre d'ID uniques : 305
Doublons ID : 0
Valeurs manquantes ID : 0

Aperçu du dataset d'orientation initiale :


,ins_CreatedAt,IDENTIFICATION,ins_Année de naissance,ins_Genre,ins_Région d'origine,ins_Filière,ins_Niveau d'étude,ins_Statut actuel,ins_Niveau informatique,ins_Niveau Excel,ins_Niveau Power BI,ins_Niveau Python,ins_Niveau IA,ins_Motivation,ins_Objectif pro,ins_Disponibilité,ins_Source info,ins_annee_naissance_originale,ins_annee_naissance,ins_age,init_CreatedAt,init_Date du test,init_Q1 - Resume ventes Excel,init_Q2 - Import CSV Excel,init_Q3 - Modele donnees Excel,init_Q4 - Mauvaise pratique visu Excel,init_Q5 - 2e grande valeur Excel,init_Q6 - Mesure vs Colonne DAX,init_Q7 - CA annee precedente DAX,init_Q8 - Vue Modele Power BI,init_Q9 - Acces directeurs regionaux,init_Q10 - 12 commerciaux 3 indicateurs,init_Q11 - Bibliotheque CSV Python,init_Q12 - Overfitting Underfitting,init_Q13 - Segmentation 50000 clients,init_Q14 - Deployer modele Python API,init_Q15 - Valeurs manquantes 30pc,init_Q16 - Role system prompt LLM,init_Q17 - Assistant IA PDF financiers,init_Q18 - Role embedding dans RAG,init_Q19 - Agent IA selection outil,init_Q20 - Sortie fiable LLM tableau
0,2026-03-31 23:34:20+00:00,DIEG2026-001,2007,Masculin,Analamanga,Informatique / Genie logiciel,Licence 2 (L2),Etudiant(e) a plein temps,Intermediaire,Avance,Intermediaire,Intermediaire,Debutant,"[""Evoluer dans mon poste""]",Data Scientist,"[""En semaine - Matin""]",LinkedIn,2007,2007.0,19.0,2026-03-31 23:36:12+00:00,2026-03-31 23:36:11+00:00,A. Filtre automatique,C. Formules SI imbriquees,D. Formules RECHERCHEV,B. Limiter les couleurs,D. RANG(plage;2),C. Mesure=visualisation Colonne=stockee,B. CALCULATE(SUM SAMEPERIODLASTYEAR),D. Definir relations entre tables,C. Rapport separe par directeur,D. Radar ou tableau matriciel conditionnel,E. Je ne sais pas,C. Underfitting,C. Regression lineaire,C. FastAPI ou Flask + joblib ou pickle,D. Supprimer toutes les lignes,E. Je ne sais pas,B. RAG - indexer et recuperer dynamiquement,C. Compresser les fichiers PDF,B. Ordre dans lequel les outils sont listes,C. Augmenter la temperature
1,2026-04-01 00:36:16+00:00,DIEG2026-002,20002,Masculin,Atsimo-Atsinanana,Polytechnique / Genie civil / Genie industriel,Master 1 (M1),Diplome(e) en recherche d emploi,Avance,Avance,Avance,Avance,Intermediaire,"[""Acquerir des competences techniques""]",Developpeur Python / IA,"[""En semaine - Apres-midi""]",Ami(e),20002,NaN,NaN,2026-04-01 00:37:59+00:00,2026-04-01 00:38:00+00:00,C. Formule SOMME(),A. Macros VBA manuelles,C. Power Pivot avec relations,C. Etiquettes de donnees claires,C. MAX(SI(...)),C. Mesure=visualisation Colonne=stockee,E. Je ne sais pas,B. Publier sur Power BI Service,B. Row-Level Security (RLS),B. Graphique en secteurs,A. matplotlib,C. Underfitting,C. Regression lineaire,D. Envoyer par e-mail,D. Supprimer toutes les lignes,E. Je ne sais pas,B. RAG - indexer et recuperer dynamiquement,B. Authentifier utilisateur,E. Je ne sais pas,D. JSON structure avec schema defini
2,2026-04-01 03:01:14+00:00,DIEG2026-003,2001,Feminin,Diana,Droit / Sciences politiques,Master 2 (M2),Diplome(e) en recherche d emploi,Aucun,Aucun,Aucun,Aucun,Aucun,"[""Ameliorer mon employabilite"", "" Acquerir des...",Entrepreneur,"[""En semaine - Matin""]",Facebook,2001,2001.0,25.0,2026-04-03 13:28:48+00:00,2026-04-03 13:28:47+00:00,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas
3,2026-04-01 03:32:24+00:00,DIEG2026-004,2003,Masculin,Diana,Autre,Master 2 (M2),Etudiant(e) en fin d etudes,Intermediaire,Intermediaire,Debutant,Intermediaire,Intermediaire,"[""Acquerir des competences techniques"", "" Amel...",Data Scientist,"[""Flexible""]",NaN,2003,2003.0,23.0,2026-04-01 03:43:02+00:00,2026-04-01 03:42:58+00:00,B. Tableau croise dynamique (TCD),D. Power Query,C. Po

Cellule 15 corrigée — Dataset de prédiction finale pré-scoring

Ce dataset fusionne:
Inscriptions préparées
+ Test initial préparé
+ Tests finaux préparés

In [21]:
# ============================================================
# 1.14. Construction du dataset de prédiction finale pré-scoring
# Inscriptions + test initial + tests finaux
# VERSION CORRIGÉE ET CONTRÔLÉE
# ============================================================

# Vérification des identifiants avant fusion
ids_inscriptions = set(df_inscriptions_pref["IDENTIFICATION"])
ids_initial = set(df_test_initial_pref["IDENTIFICATION"])
ids_finaux = set(df_finaux_pref["IDENTIFICATION"])

ids_communs_prediction = ids_inscriptions & ids_initial & ids_finaux

print("Nombre d'ID inscriptions :", len(ids_inscriptions))
print("Nombre d'ID test initial :", len(ids_initial))
print("Nombre d'ID tests finaux :", len(ids_finaux))
print("Nombre d'ID communs inscriptions + initial + final :", len(ids_communs_prediction))


# Fusion contrôlée : inscriptions + test initial
df_temp_prediction = df_inscriptions_pref.merge(
    df_test_initial_pref,
    on="IDENTIFICATION",
    how="inner",
    validate="one_to_one"
)

# Fusion contrôlée avec tests finaux
df_dataset_prediction_finale = df_temp_prediction.merge(
    df_finaux_pref,
    on="IDENTIFICATION",
    how="inner",
    validate="one_to_one"
)

afficher_shape("Dataset prédiction finale pré-scoring", df_dataset_prediction_finale)

print("\nVérification après fusion :")
print("Nombre de lignes :", len(df_dataset_prediction_finale))
print("Nombre d'ID uniques :", df_dataset_prediction_finale["IDENTIFICATION"].nunique())
print("Doublons ID :", df_dataset_prediction_finale["IDENTIFICATION"].duplicated().sum())
print("Valeurs manquantes ID :", df_dataset_prediction_finale["IDENTIFICATION"].isna().sum())

print("\nRépartition par parcours final :")
if "final_parcours_test_final" in df_dataset_prediction_finale.columns:
    display(df_dataset_prediction_finale["final_parcours_test_final"].value_counts())
else:
    print("Colonne final_parcours_test_final non trouvée.")

print("\nAperçu du dataset de prédiction finale pré-scoring :")
display(df_dataset_prediction_finale.head())

Nombre d'ID inscriptions : 472
Nombre d'ID test initial : 305
Nombre d'ID tests finaux : 206
Nombre d'ID communs inscriptions + initial + final : 128
Dataset prédiction finale pré-scoring    : 128 lignes | 64 colonnes

Vérification après fusion :
Nombre de lignes : 128
Nombre d'ID uniques : 128
Doublons ID : 0
Valeurs manquantes ID : 0

Répartition par parcours final :


final_parcours_test_final
BI    42
IA    40
DS    28
DA    18
Name: count, dtype: int64


Aperçu du dataset de prédiction finale pré-scoring :


,ins_CreatedAt,IDENTIFICATION,ins_Année de naissance,ins_Genre,ins_Région d'origine,ins_Filière,ins_Niveau d'étude,ins_Statut actuel,ins_Niveau informatique,ins_Niveau Excel,ins_Niveau Power BI,ins_Niveau Python,ins_Niveau IA,ins_Motivation,ins_Objectif pro,ins_Disponibilité,ins_Source info,ins_annee_naissance_originale,ins_annee_naissance,ins_age,init_CreatedAt,init_Date du test,init_Q1 - Resume ventes Excel,init_Q2 - Import CSV Excel,init_Q3 - Modele donnees Excel,init_Q4 - Mauvaise pratique visu Excel,init_Q5 - 2e grande valeur Excel,init_Q6 - Mesure vs Colonne DAX,init_Q7 - CA annee precedente DAX,init_Q8 - Vue Modele Power BI,init_Q9 - Acces directeurs regionaux,init_Q10 - 12 commerciaux 3 indicateurs,init_Q11 - Bibliotheque CSV Python,init_Q12 - Overfitting Underfitting,init_Q13 - Segmentation 50000 clients,init_Q14 - Deployer modele Python API,init_Q15 - Valeurs manquantes 30pc,init_Q16 - Role system prompt LLM,init_Q17 - Assistant IA PDF financiers,init_Q18 - Role embedding dans RAG,init_Q19 - Agent IA selection outil,init_Q20 - Sortie fiable LLM tableau,final_RowId,final_CreatedAt,final_InscriptionId,final_Filiere,final_Niveau d'Etude,final_Q01,final_Q02,final_Q03,final_Q04,final_Q05,final_Q06,final_Q07,final_Q08,final_Q09,final_Q10,final_Q11,final_Q12,final_Q13,final_Q14,final_Q15,final_parcours_test_final,final_feuille_source_finale
0,2026-04-01 03:01:14+00:00,DIEG2026-003,2001,Feminin,Diana,Droit / Sciences politiques,Master 2 (M2),Diplome(e) en recherche d emploi,Aucun,Aucun,Aucun,Aucun,Aucun,"[""Ameliorer mon employabilite"", "" Acquerir des...",Entrepreneur,"[""En semaine - Matin""]",Facebook,2001,2001.0,25.0,2026-04-03 13:28:48+00:00,2026-04-03 13:28:47+00:00,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,15,2026-06-04 20:03:57+00:00,3,Droit / Sciences politiques,Master 2 (M2),"B) Extraire le fichier, nettoyer/transformer l...",A) Les mesures et cles vers les dimensions,B) Workspace / Espace de travail > Skills,A) Administration > Reglages > Integrations,C) Administration > Fonctions > Nouvelle fonction,D) Valider et structurer les donnees recues et...,D) Generer des graphiques interactifs ou expor...,C) Stocker et exposer par API les contenus/rap...,C) Les prompts temporaires non publies,C) Afficher les rapports et declencher/recuper...,C) Des nodes connectes qui executent des actio...,"B) Lancer l'ETL, appeler les API, publier dans...",A) Piloter ou modifier des workflows et integr...,"D) Verifier l'endpoint/API appele, le slug/id ...",D) Administration > Reglages > Connexion,IA,FINAL-TEST-IA
1,2026-04-01 06:30:52+00:00,DIEG2026-007,2002,Masculin,Sofia,Autre,Master 2 (M2),Etudiant(e) en fin d etudes,Intermediaire,Debutant,Aucun,Intermediaire,Debutant,"[""Acquerir des competences techniques"", "" Real...",Developpeur Python / IA,"[""En semaine - Soir"", "" Week-end""]",WhatsApp,2002,2002.0,24.0,2026-04-01 06:41:29+00:00,2026-04-01 06:41:26+00:00,B. Tableau croise dynamique (TCD),C. Formules SI imbriquees,B. TCD simple,E. Je ne sais pas,B. GRANDE.VALEUR(plage;2),E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,B. Row-Level Security (RLS),C. 12 courbes separees,C. pandas,D. Overfitting,A. Reseau de neurones recurrent,C. FastAPI ou Flask + joblib ou pickle,C. Imputer avec mediane ou modele,A. Connecter a une base de donnees,D. Connaissances integrees du LLM uniquement,D. Transformer texte en vecteurs numeriques,E. Je ne sais pas,E. Je ne sais pas,28,2026-06-05 02:35:06+00:00,7,Autre,Master 2 (M2),"B) Extraire le fichier, nettoyer/transformer l...",A) Les mesures et cles vers les dimensions,B) Workspace / Espace de travail > Skills,A) Administration > Reglages > Integrations,C) Administration > Fonctio

Cellule 16 corrigée — Dataset longitudinal complet

In [22]:
# ============================================================
# 1.15. Construction du dataset longitudinal complet pré-scoring
# Inscriptions + test initial + tests intermédiaires + tests finaux
# VERSION CORRIGÉE ET CONTRÔLÉE
# ============================================================

# Vérification des identifiants avant fusion
ids_inscriptions = set(df_inscriptions_pref["IDENTIFICATION"])
ids_initial = set(df_test_initial_pref["IDENTIFICATION"])
ids_intermediaires = set(df_intermediaires_pref["IDENTIFICATION"])
ids_finaux = set(df_finaux_pref["IDENTIFICATION"])

ids_communs_longitudinal = (
    ids_inscriptions
    & ids_initial
    & ids_intermediaires
    & ids_finaux
)

print("Nombre d'ID inscriptions :", len(ids_inscriptions))
print("Nombre d'ID test initial :", len(ids_initial))
print("Nombre d'ID tests intermédiaires :", len(ids_intermediaires))
print("Nombre d'ID tests finaux :", len(ids_finaux))
print("Nombre d'ID communs aux 4 sources :", len(ids_communs_longitudinal))


# Fusion contrôlée des 4 sources
df_dataset_longitudinal = (
    df_inscriptions_pref
    .merge(
        df_test_initial_pref,
        on="IDENTIFICATION",
        how="inner",
        validate="one_to_one"
    )
    .merge(
        df_intermediaires_pref,
        on="IDENTIFICATION",
        how="inner",
        validate="one_to_one"
    )
    .merge(
        df_finaux_pref,
        on="IDENTIFICATION",
        how="inner",
        validate="one_to_one"
    )
)

afficher_shape("Dataset longitudinal complet pré-scoring", df_dataset_longitudinal)

print("\nVérification après fusion :")
print("Nombre de lignes :", len(df_dataset_longitudinal))
print("Nombre d'ID uniques :", df_dataset_longitudinal["IDENTIFICATION"].nunique())
print("Doublons ID :", df_dataset_longitudinal["IDENTIFICATION"].duplicated().sum())
print("Valeurs manquantes ID :", df_dataset_longitudinal["IDENTIFICATION"].isna().sum())


# Répartition par parcours intermédiaire
print("\nRépartition par parcours intermédiaire :")
if "mi_parcours_test_intermediaire" in df_dataset_longitudinal.columns:
    display(df_dataset_longitudinal["mi_parcours_test_intermediaire"].value_counts())
else:
    print("Colonne mi_parcours_test_intermediaire non trouvée.")


# Répartition par parcours final
print("\nRépartition par parcours final :")
if "final_parcours_test_final" in df_dataset_longitudinal.columns:
    display(df_dataset_longitudinal["final_parcours_test_final"].value_counts())
else:
    print("Colonne final_parcours_test_final non trouvée.")


# Contrôle de cohérence entre parcours intermédiaire et parcours final
if (
    "mi_parcours_test_intermediaire" in df_dataset_longitudinal.columns
    and "final_parcours_test_final" in df_dataset_longitudinal.columns
):
    df_dataset_longitudinal["coherence_parcours_intermediaire_final"] = (
        df_dataset_longitudinal["mi_parcours_test_intermediaire"]
        == df_dataset_longitudinal["final_parcours_test_final"]
    )
    
    print("\nCohérence parcours intermédiaire / parcours final :")
    display(df_dataset_longitudinal["coherence_parcours_intermediaire_final"].value_counts())
    
    df_parcours_incoherents = df_dataset_longitudinal[
        df_dataset_longitudinal["coherence_parcours_intermediaire_final"] == False
    ][[
        "IDENTIFICATION",
        "mi_parcours_test_intermediaire",
        "final_parcours_test_final"
    ]].copy()
    
    print("\nNombre de parcours incohérents :", len(df_parcours_incoherents))
    
    if len(df_parcours_incoherents) > 0:
        display(df_parcours_incoherents.head(30))
else:
    df_parcours_incoherents = pd.DataFrame()


print("\nAperçu du dataset longitudinal complet pré-scoring :")
display(df_dataset_longitudinal.head())

Nombre d'ID inscriptions : 472
Nombre d'ID test initial : 305
Nombre d'ID tests intermédiaires : 196
Nombre d'ID tests finaux : 206
Nombre d'ID communs aux 4 sources : 125
Dataset longitudinal complet pré-scoring : 125 lignes | 83 colonnes

Vérification après fusion :
Nombre de lignes : 125
Nombre d'ID uniques : 125
Doublons ID : 0
Valeurs manquantes ID : 0

Répartition par parcours intermédiaire :


mi_parcours_test_intermediaire
BI    42
IA    40
DS    26
DA    17
Name: count, dtype: int64


Répartition par parcours final :


final_parcours_test_final
BI    42
IA    40
DS    26
DA    17
Name: count, dtype: int64


Cohérence parcours intermédiaire / parcours final :


coherence_parcours_intermediaire_final
True    125
Name: count, dtype: int64


Nombre de parcours incohérents : 0

Aperçu du dataset longitudinal complet pré-scoring :


,ins_CreatedAt,IDENTIFICATION,ins_Année de naissance,ins_Genre,ins_Région d'origine,ins_Filière,ins_Niveau d'étude,ins_Statut actuel,ins_Niveau informatique,ins_Niveau Excel,ins_Niveau Power BI,ins_Niveau Python,ins_Niveau IA,ins_Motivation,ins_Objectif pro,ins_Disponibilité,ins_Source info,ins_annee_naissance_originale,ins_annee_naissance,ins_age,init_CreatedAt,init_Date du test,init_Q1 - Resume ventes Excel,init_Q2 - Import CSV Excel,init_Q3 - Modele donnees Excel,init_Q4 - Mauvaise pratique visu Excel,init_Q5 - 2e grande valeur Excel,init_Q6 - Mesure vs Colonne DAX,init_Q7 - CA annee precedente DAX,init_Q8 - Vue Modele Power BI,init_Q9 - Acces directeurs regionaux,init_Q10 - 12 commerciaux 3 indicateurs,init_Q11 - Bibliotheque CSV Python,init_Q12 - Overfitting Underfitting,init_Q13 - Segmentation 50000 clients,init_Q14 - Deployer modele Python API,init_Q15 - Valeurs manquantes 30pc,init_Q16 - Role system prompt LLM,init_Q17 - Assistant IA PDF financiers,init_Q18 - Role embedding dans RAG,init_Q19 - Agent IA selection outil,init_Q20 - Sortie fiable LLM tableau,mi_Id,mi_CreatedAt,mi_Q01,mi_Q02,mi_Q03,mi_Q04,mi_Q05,mi_Q06,mi_Q07,mi_Q08,mi_Q09,mi_Q10,mi_Q11,mi_Q12,mi_Q13,mi_Q14,mi_Q15,mi_parcours_test_intermediaire,mi_feuille_source_intermediaire,final_RowId,final_CreatedAt,final_InscriptionId,final_Filiere,final_Niveau d'Etude,final_Q01,final_Q02,final_Q03,final_Q04,final_Q05,final_Q06,final_Q07,final_Q08,final_Q09,final_Q10,final_Q11,final_Q12,final_Q13,final_Q14,final_Q15,final_parcours_test_final,final_feuille_source_finale,coherence_parcours_intermediaire_final
0,2026-04-01 03:01:14+00:00,DIEG2026-003,2001,Feminin,Diana,Droit / Sciences politiques,Master 2 (M2),Diplome(e) en recherche d emploi,Aucun,Aucun,Aucun,Aucun,Aucun,"[""Ameliorer mon employabilite"", "" Acquerir des...",Entrepreneur,"[""En semaine - Matin""]",Facebook,2001,2001.0,25.0,2026-04-03 13:28:48+00:00,2026-04-03 13:28:47+00:00,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,37,2026-05-29 05:41:30+00:00,C) Parce qu'il predit le token suivant le plus...,C) Le Lost in the Middle — les LLMs retiennent...,C) Chain-of-Thought (CoT) — Reflechis etape pa...,D) temperature = 0 — le modele choisit toujour...,D) Function Calling / Tool Use — l'API force l...,D) Garde-fous et interdictions — les comportem...,B) 200 a 500 mots — assez detaille pour etre p...,"A) Le Skill utilise uniquement Python, le Syst...",C) Les procedures detaillees de calcul de bull...,C) Le Tool Python permet au LLM d'obtenir des ...,C) Model Context Protocol — standard universel...,B) L'environnement de developpement IA (Claude...,C) Via le MCP Server N8N (mcp-n8n) — Antigravi...,"C) RAG — decouper le PDF en chunks, vectoriser...",C) A transformer chaque chunk de texte en vect...,IA,MI-TEST-IA,15,2026-06-04 20:03:57+00:00,3,Droit / Sciences politiques,Master 2 (M2),"B) Extraire le fichier, nettoyer/transformer l...",A) Les mesures et cles vers les dimensions,B) Workspace / Espace de travail > Skills,A) Administration > Reglages > Integrations,C) Administration > Fonctions > Nouvelle fonction,D) Valider et structurer les donnees recues et...,D) Generer des graphiques interactifs ou expor...,C) Stocker et exposer par API les contenus/rap...,C) Les prompts temporaires non publies,C) Afficher les rapports et declencher/recuper...,C) Des nodes connectes qui executent des actio...,"B) Lancer l'ETL, appeler les API, publier dans...",A) Piloter ou modifier des workflows et integr...,"D) Verifier l'endpoint/API appele, le slug/id ...",D) Administration > Reglages > Connexion,IA,FINAL-TEST-IA,True
1,2026-04-01 06:30:52+00:00,DIEG2026-007,2002,Masculin,Sofia,Autre,Master 2 (M2),Etudiant(e) en fin d e

Cellule 17 corrigée — Couverture après préparation

In [23]:
# ============================================================
# 1.16. Couverture après préparation
# VERSION CORRIGÉE ET DÉTAILLÉE
# ============================================================

def set_identifiants(df):
    """
    Retourne l'ensemble des identifiants non manquants d'un DataFrame.
    """
    return set(df["IDENTIFICATION"].dropna())


# Ensembles d'identifiants par source
set_inscriptions = set_identifiants(df_inscriptions)
set_initial = set_identifiants(df_test_initial)
set_intermediaires = set_identifiants(df_tests_intermediaires)
set_finaux = set_identifiants(df_tests_finaux)


# Calcul des couvertures principales
couverture_step1 = {
    "identifiants_inscriptions": len(set_inscriptions),
    "identifiants_test_initial": len(set_initial),
    "identifiants_tests_intermediaires": len(set_intermediaires),
    "identifiants_tests_finaux": len(set_finaux),

    "inscriptions_et_test_initial": len(set_inscriptions & set_initial),
    "inscriptions_et_tests_intermediaires": len(set_inscriptions & set_intermediaires),
    "inscriptions_et_tests_finaux": len(set_inscriptions & set_finaux),

    "test_initial_et_tests_intermediaires": len(set_initial & set_intermediaires),
    "test_initial_et_tests_finaux": len(set_initial & set_finaux),
    "tests_intermediaires_et_tests_finaux": len(set_intermediaires & set_finaux),

    "inscriptions_test_initial_tests_finaux": len(set_inscriptions & set_initial & set_finaux),
    "inscriptions_test_initial_tests_intermediaires_tests_finaux": len(
        set_inscriptions & set_initial & set_intermediaires & set_finaux
    ),

    "test_initial_sans_test_final": len((set_inscriptions & set_initial) - set_finaux),
    "test_final_sans_test_initial": len((set_inscriptions & set_finaux) - set_initial),
    "test_intermediaire_sans_test_final": len((set_inscriptions & set_intermediaires) - set_finaux),
    "test_final_sans_test_intermediaire": len((set_inscriptions & set_finaux) - set_intermediaires),
}


df_couverture_step1 = pd.DataFrame(
    list(couverture_step1.items()),
    columns=["indicateur", "valeur"]
)

display(df_couverture_step1)

,indicateur,valeur
0,identifiants_inscriptions,472
1,identifiants_test_initial,305
2,identifiants_tests_intermediaires,196
3,identifiants_tests_finaux,206
4,inscriptions_et_test_initial,305
5,inscriptions_et_tests_intermediaires,141
6,inscriptions_et_tests_finaux,146
7,test_initial_et_tests_intermediaires,125
8,test_initial_et_tests_finaux,128
9,tests_intermediaires_et_tests_finaux,196


Cellule 17 bis — Statut de présence de chaque apprenant

In [24]:
# ============================================================
# 1.16 bis. Matrice de présence des apprenants par source
# ============================================================

tous_les_ids = sorted(
    set_inscriptions
    | set_initial
    | set_intermediaires
    | set_finaux
)

df_presence_sources = pd.DataFrame({
    "IDENTIFICATION": tous_les_ids
})

df_presence_sources["present_inscription"] = df_presence_sources["IDENTIFICATION"].isin(set_inscriptions)
df_presence_sources["present_test_initial"] = df_presence_sources["IDENTIFICATION"].isin(set_initial)
df_presence_sources["present_test_intermediaire"] = df_presence_sources["IDENTIFICATION"].isin(set_intermediaires)
df_presence_sources["present_test_final"] = df_presence_sources["IDENTIFICATION"].isin(set_finaux)


def definir_statut_presence(row):
    """
    Définit le statut de disponibilité des données pour chaque apprenant.
    """
    ins = row["present_inscription"]
    init = row["present_test_initial"]
    mi = row["present_test_intermediaire"]
    final = row["present_test_final"]
    
    if ins and init and mi and final:
        return "complet_4_sources"
    elif ins and init and final and not mi:
        return "inscription_initial_final_sans_intermediaire"
    elif ins and init and not final:
        return "orientation_initiale_sans_final"
    elif ins and final and not init:
        return "final_sans_initial"
    elif ins and not init and not mi and not final:
        return "inscription_seule"
    elif not ins:
        return "hors_inscription"
    else:
        return "autre_cas"


df_presence_sources["statut_presence"] = df_presence_sources.apply(
    definir_statut_presence,
    axis=1
)

print("Répartition des statuts de présence :")
display(df_presence_sources["statut_presence"].value_counts())

print("\nAperçu de la matrice de présence :")
display(df_presence_sources.head(20))

Répartition des statuts de présence :


statut_presence
orientation_initiale_sans_final                 177
inscription_seule                               149
complet_4_sources                               125
hors_inscription                                 60
final_sans_initial                               18
inscription_initial_final_sans_intermediaire      3
Name: count, dtype: int64


Aperçu de la matrice de présence :


,IDENTIFICATION,present_inscription,present_test_initial,present_test_intermediaire,present_test_final,statut_presence
0,DIEG2026-001,True,True,False,False,orientation_initiale_sans_final
1,DIEG2026-002,True,True,False,False,orientation_initiale_sans_final
2,DIEG2026-003,True,True,True,True,complet_4_sources
3,DIEG2026-004,True,True,False,False,orientation_initiale_sans_final
4,DIEG2026-005,True,True,False,False,orientation_initiale_sans_final
5,DIEG2026-006,True,True,False,False,orientation_initiale_sans_final
6,DIEG2026-007,True,True,True,True,complet_4_sources
7,DIEG2026-008,True,True,True,True,complet_4_sources
8,DIEG2026-009,True,True,True,True,complet_4_sources
9,DIEG2026-010,True,False,True,True,final_sans_initial


Cellule 18 corrigée — Export des fichiers préparés

In [25]:
# ============================================================
# 1.17. Export des fichiers préparés
# VERSION CORRIGÉE ET COMPLÈTE
# ============================================================

OUTPUT_DIR = Path("outputs/01_integration_preparation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def exporter_csv(df, nom_fichier):
    """
    Exporte un DataFrame en CSV avec encodage compatible Excel.
    """
    chemin = OUTPUT_DIR / nom_fichier
    df.to_csv(
        chemin,
        index=False,
        encoding="utf-8-sig"
    )
    print(f"Exporté : {nom_fichier} | {df.shape[0]} lignes | {df.shape[1]} colonnes")


# ============================================================
# 1. Sources préparées
# ============================================================

exporter_csv(
    df_inscriptions,
    "01_inscriptions_preparees.csv"
)

exporter_csv(
    df_test_initial,
    "02_test_initial_prepare.csv"
)

exporter_csv(
    df_tests_intermediaires,
    "03_tests_intermediaires_prepares.csv"
)

exporter_csv(
    df_tests_finaux,
    "04_tests_finaux_prepares.csv"
)


# ============================================================
# 2. Datasets construits
# ============================================================

exporter_csv(
    df_dataset_orientation_initiale,
    "05_dataset_orientation_initiale.csv"
)

exporter_csv(
    df_dataset_prediction_finale,
    "06_dataset_prediction_finale_pre_scoring.csv"
)

exporter_csv(
    df_dataset_longitudinal,
    "07_dataset_longitudinal_pre_scoring.csv"
)


# ============================================================
# 3. Rapports de contrôle
# ============================================================

exporter_csv(
    df_resume_ids_step1,
    "08_resume_identifiants_step1.csv"
)

exporter_csv(
    df_couverture_step1,
    "09_couverture_apres_preparation.csv"
)

exporter_csv(
    df_presence_sources,
    "10_presence_sources_apprenants.csv"
)


# ============================================================
# 4. Mappings des questions
# ============================================================

exporter_csv(
    df_mapping_questions_intermediaires,
    "11_mapping_questions_intermediaires.csv"
)

exporter_csv(
    df_mapping_questions_finales,
    "12_mapping_questions_finales.csv"
)


# ============================================================
# 5. Fichiers d'anomalies ou de lignes isolées
# ============================================================

if "df_annees_aberrantes" in globals() and not df_annees_aberrantes.empty:
    exporter_csv(
        df_annees_aberrantes,
        "13_annees_naissance_aberrantes_neutralisees.csv"
    )

if "df_test_initial_sans_id" in globals() and not df_test_initial_sans_id.empty:
    exporter_csv(
        df_test_initial_sans_id,
        "14_lignes_test_initial_sans_identifiant.csv"
    )

if "df_tests_intermediaires_sans_id" in globals() and not df_tests_intermediaires_sans_id.empty:
    exporter_csv(
        df_tests_intermediaires_sans_id,
        "15_lignes_tests_intermediaires_sans_identifiant.csv"
    )

if "df_tests_finaux_sans_id" in globals() and not df_tests_finaux_sans_id.empty:
    exporter_csv(
        df_tests_finaux_sans_id,
        "16_lignes_tests_finaux_sans_identifiant.csv"
    )

if "df_parcours_incoherents" in globals() and not df_parcours_incoherents.empty:
    exporter_csv(
        df_parcours_incoherents,
        "17_parcours_intermediaire_final_incoherents.csv"
    )

if "df_doublons_step1" in globals() and not df_doublons_step1.empty:
    exporter_csv(
        df_doublons_step1,
        "18_doublons_identifiants_step1.csv"
    )


# ============================================================
# 6. Liste finale des fichiers générés
# ============================================================

print("\nExports terminés dans :", OUTPUT_DIR)
print("\nListe des fichiers générés :")

for fichier in sorted(OUTPUT_DIR.iterdir()):
    print("-", fichier.name)

Exporté : 01_inscriptions_preparees.csv | 472 lignes | 20 colonnes
Exporté : 02_test_initial_prepare.csv | 305 lignes | 23 colonnes
Exporté : 03_tests_intermediaires_prepares.csv | 196 lignes | 20 colonnes
Exporté : 04_tests_finaux_prepares.csv | 206 lignes | 23 colonnes
Exporté : 05_dataset_orientation_initiale.csv | 305 lignes | 42 colonnes
Exporté : 06_dataset_prediction_finale_pre_scoring.csv | 128 lignes | 64 colonnes
Exporté : 07_dataset_longitudinal_pre_scoring.csv | 125 lignes | 84 colonnes
Exporté : 08_resume_identifiants_step1.csv | 4 lignes | 6 colonnes
Exporté : 09_couverture_apres_preparation.csv | 16 lignes | 2 colonnes
Exporté : 10_presence_sources_apprenants.csv | 532 lignes | 6 colonnes
Exporté : 11_mapping_questions_intermediaires.csv | 60 lignes | 4 colonnes
Exporté : 12_mapping_questions_finales.csv | 60 lignes | 4 colonnes
Exporté : 13_annees_naissance_aberrantes_neutralisees.csv | 9 lignes | 3 colonnes
Exporté : 14_lignes_test_initial_sans_identifiant.csv | 10 lig

01_inscriptions_preparees.csv                    : 472 lignes
02_test_initial_prepare.csv                      : 305 lignes
03_tests_intermediaires_prepares.csv             : 196 lignes
04_tests_finaux_prepares.csv                     : 206 lignes
05_dataset_orientation_initiale.csv              : 305 lignes
06_dataset_prediction_finale_pre_scoring.csv     : 128 lignes
07_dataset_longitudinal_pre_scoring.csv          : 125 lignes

Cellule 19 corrigée — Rapport automatique de l’Étape 1

In [26]:
# ============================================================
# 1.18. Rapport automatique de l'Étape 1
# VERSION CORRIGÉE ET COMPLÈTE
# ============================================================

from datetime import datetime
import json


def shape_dict(df):
    """
    Retourne la forme d'un DataFrame sous forme JSON-compatible.
    """
    return {
        "nb_lignes": int(df.shape[0]),
        "nb_colonnes": int(df.shape[1])
    }


def safe_len(variable_name):
    """
    Retourne la longueur d'une variable si elle existe, sinon 0.
    """
    if variable_name in globals():
        return int(len(globals()[variable_name]))
    return 0


def safe_shape(variable_name):
    """
    Retourne la shape d'un DataFrame s'il existe.
    """
    if variable_name in globals():
        return shape_dict(globals()[variable_name])
    return None


rapport_step1 = {
    "etape": "Étape 1 — Intégration et préparation des données",
    "date_generation": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),

    "sources_preparees": {
        "inscriptions_preparees": shape_dict(df_inscriptions),
        "test_initial_prepare": shape_dict(df_test_initial),
        "tests_intermediaires_prepares": shape_dict(df_tests_intermediaires),
        "tests_finaux_prepares": shape_dict(df_tests_finaux)
    },

    "datasets_construits": {
        "dataset_orientation_initiale": shape_dict(df_dataset_orientation_initiale),
        "dataset_prediction_finale_pre_scoring": shape_dict(df_dataset_prediction_finale),
        "dataset_longitudinal_pre_scoring": shape_dict(df_dataset_longitudinal)
    },

    "controles_qualite": {
        "nb_annees_naissance_aberrantes_neutralisees": safe_len("df_annees_aberrantes"),
        "nb_lignes_test_initial_sans_id": safe_len("df_test_initial_sans_id"),
        "nb_lignes_tests_intermediaires_sans_id": safe_len("df_tests_intermediaires_sans_id"),
        "nb_lignes_tests_finaux_sans_id": safe_len("df_tests_finaux_sans_id"),
        "nb_lignes_doublons_identifiants": safe_len("df_doublons_step1"),
        "nb_parcours_incoherents": safe_len("df_parcours_incoherents")
    },

    "couverture": {
        row["indicateur"]: int(row["valeur"])
        for _, row in df_couverture_step1.iterrows()
    },

    "nomenclature_officielle": {
        "terme_general": "apprenant",
        "niveaux_competence": [
            "Débutant",
            "Intermédiaire",
            "Avancé"
        ]
    },

    "decision": {
        "statut": "Étape 1 validée",
        "commentaire": (
            "Les sources ont été préparées, les identifiants ont été harmonisés, "
            "les lignes sans identifiant ont été isolées, les années de naissance aberrantes "
            "ont été neutralisées et les datasets pré-scoring ont été construits."
        ),
        "prochaine_etape": "Étape 2 — Scoring pédagogique et calcul des niveaux"
    }
}


# Export JSON du rapport
chemin_rapport_json = OUTPUT_DIR / "rapport_step1_integration_preparation.json"

with open(chemin_rapport_json, "w", encoding="utf-8") as f:
    json.dump(
        rapport_step1,
        f,
        ensure_ascii=False,
        indent=4
    )


# Export Markdown lisible
rapport_markdown = f"""# Rapport automatique — Étape 1

## Étape
Étape 1 — Intégration et préparation des données

## Date de génération
{rapport_step1["date_generation"]}

## 1. Sources préparées

| Source | Lignes | Colonnes |
|---|---:|---:|
| Inscriptions préparées | {df_inscriptions.shape[0]} | {df_inscriptions.shape[1]} |
| Test initial préparé | {df_test_initial.shape[0]} | {df_test_initial.shape[1]} |
| Tests intermédiaires préparés | {df_tests_intermediaires.shape[0]} | {df_tests_intermediaires.shape[1]} |
| Tests finaux préparés | {df_tests_finaux.shape[0]} | {df_tests_finaux.shape[1]} |

## 2. Datasets construits

| Dataset | Lignes | Colonnes | Usage |
|---|---:|---:|---|
| Dataset d’orientation initiale | {df_dataset_orientation_initiale.shape[0]} | {df_dataset_orientation_initiale.shape[1]} | Recommandation initiale de parcours |
| Dataset de prédiction finale pré-scoring | {df_dataset_prediction_finale.shape[0]} | {df_dataset_prediction_finale.shape[1]} | Prédiction des performances finales |
| Dataset longitudinal pré-scoring | {df_dataset_longitudinal.shape[0]} | {df_dataset_longitudinal.shape[1]} | Analyse complète inscription + initial + intermédiaire + final |

## 3. Contrôles qualité

| Contrôle | Valeur |
|---|---:|
| Années de naissance aberrantes neutralisées | {safe_len("df_annees_aberrantes")} |
| Lignes du test initial sans identifiant | {safe_len("df_test_initial_sans_id")} |
| Lignes des tests intermédiaires sans identifiant | {safe_len("df_tests_intermediaires_sans_id")} |
| Lignes des tests finaux sans identifiant | {safe_len("df_tests_finaux_sans_id")} |
| Doublons d’identifiants détectés | {safe_len("df_doublons_step1")} |
| Parcours intermédiaire/final incohérents | {safe_len("df_parcours_incoherents")} |

## 4. Nomenclature officielle

Dans ce mémoire, le terme **apprenant** est utilisé de manière générale. Il ne désigne pas nécessairement un étudiant, mais toute personne bénéficiant du dispositif de formation.

Les niveaux de compétence et les clusters sont définis selon trois catégories :

- Débutant
- Intermédiaire
- Avancé

## 5. Décision

**Statut : Étape 1 validée.**

Les sources de données sont maintenant préparées, harmonisées et exportées. Les datasets pré-scoring sont disponibles pour la suite du projet.

## 6. Prochaine étape

**Étape 2 — Scoring pédagogique et calcul des niveaux.**
"""

chemin_rapport_md = OUTPUT_DIR / "rapport_step1_integration_preparation.md"

with open(chemin_rapport_md, "w", encoding="utf-8") as f:
    f.write(rapport_markdown)


print("Rapport JSON généré :", chemin_rapport_json)
print("Rapport Markdown généré :", chemin_rapport_md)

print("\n===== SYNTHÈSE ÉTAPE 1 =====")
print("Inscriptions préparées :", df_inscriptions.shape)
print("Test initial préparé :", df_test_initial.shape)
print("Tests intermédiaires préparés :", df_tests_intermediaires.shape)
print("Tests finaux préparés :", df_tests_finaux.shape)
print("Dataset orientation initiale :", df_dataset_orientation_initiale.shape)
print("Dataset prédiction finale pré-scoring :", df_dataset_prediction_finale.shape)
print("Dataset longitudinal pré-scoring :", df_dataset_longitudinal.shape)

print("\nStatut : Étape 1 validée")
print("Prochaine étape : Étape 2 — Scoring pédagogique et calcul des niveaux")

rapport_step1

Rapport JSON généré : outputs/01_integration_preparation/rapport_step1_integration_preparation.json
Rapport Markdown généré : outputs/01_integration_preparation/rapport_step1_integration_preparation.md

===== SYNTHÈSE ÉTAPE 1 =====
Inscriptions préparées : (472, 20)
Test initial préparé : (305, 23)
Tests intermédiaires préparés : (196, 20)
Tests finaux préparés : (206, 23)
Dataset orientation initiale : (305, 42)
Dataset prédiction finale pré-scoring : (128, 64)
Dataset longitudinal pré-scoring : (125, 84)

Statut : Étape 1 validée
Prochaine étape : Étape 2 — Scoring pédagogique et calcul des niveaux


{'etape': 'Étape 1 — Intégration et préparation des données',
 'date_generation': '2026-09-09 01:06:58',
 'sources_preparees': {'inscriptions_preparees': {'nb_lignes': 472,
   'nb_colonnes': 20},
  'test_initial_prepare': {'nb_lignes': 305, 'nb_colonnes': 23},
  'tests_intermediaires_prepares': {'nb_lignes': 196, 'nb_colonnes': 20},
  'tests_finaux_prepares': {'nb_lignes': 206, 'nb_colonnes': 23}},
 'datasets_construits': {'dataset_orientation_initiale': {'nb_lignes': 305,
   'nb_colonnes': 42},
  'dataset_prediction_finale_pre_scoring': {'nb_lignes': 128,
   'nb_colonnes': 64},
  'dataset_longitudinal_pre_scoring': {'nb_lignes': 125, 'nb_colonnes': 84}},
 'controles_qualite': {'nb_annees_naissance_aberrantes_neutralisees': 9,
  'nb_lignes_test_initial_sans_id': 10,
  'nb_lignes_tests_intermediaires_sans_id': 14,
  'nb_lignes_tests_finaux_sans_id': 0,
  'nb_lignes_doublons_identifiants': 0,
  'nb_parcours_incoherents': 0},
 'couverture': {'identifiants_inscriptions': 472,
  'identifian

Sources préparées :
- Inscriptions : 472 lignes | 20 colonnes
- Test initial : 305 lignes | 23 colonnes
- Tests intermédiaires : 196 lignes | 20 colonnes
- Tests finaux : 206 lignes | 23 colonnes

Datasets construits :
- Dataset orientation initiale : 305 lignes | 42 colonnes
- Dataset prédiction finale pré-scoring : 128 lignes | 64 colonnes
- Dataset longitudinal pré-scoring : 125 lignes | 84 colonnes

Les contrôles qualité sont aussi bons :
- Années de naissance aberrantes neutralisées : 9
- Lignes du test initial sans identifiant : 10
- Lignes des tests intermédiaires sans identifiant : 14
- Lignes des tests finaux sans identifiant : 0
- Doublons d’identifiants : 0
- Parcours intermédiaire/final incohérents : 0

Étape 1 — Intégration et préparation des données : VALIDÉE